<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_08_Real_Staging_and_Vocabulary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation
import os, sys, json, csv, re, io, zipfile, hashlib, hmac, secrets, sqlite3, subprocess
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

BASE = Path('/content')
PROJECT = BASE / 'drive/MyDrive/NeuroFHIR_ICHI2027'
PRIVATE_WORK = BASE / 'ICHI2027_Phase08_PRIVATE_SESSION'
PRIVATE_WORK.mkdir(parents=True, exist_ok=True)
OUTPUT = PROJECT / 'phase08_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
SHAREABLE = OUTPUT / 'ICHI2027_Phase08_SHAREABLE.json'
SOURCE_NAME = 'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'

def file_sha(path):
    sha = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(2**20), b''):
            sha.update(chunk)
    return sha.hexdigest()

def now():
    return datetime.now(timezone.utc).isoformat()

Mounted at /content/drive


In [2]:
phase07_options = [
    PROJECT / 'phase07_outputs/ICHI2027_Phase07_SHAREABLE.json',
    BASE / 'ICHI2027_Phase07_SHAREABLE.json',
]
phase07_file = next((p for p in phase07_options if p.is_file()), None)
if phase07_file is None:
    from google.colab import files
    uploaded = files.upload()
    if 'ICHI2027_Phase07_SHAREABLE.json' not in uploaded:
        raise FileNotFoundError('ICHI2027_Phase07_SHAREABLE.json required')
    phase07_file = PRIVATE_WORK / 'ICHI2027_Phase07_SHAREABLE.json'
    phase07_file.write_bytes(uploaded['ICHI2027_Phase07_SHAREABLE.json'])
phase07 = json.loads(phase07_file.read_text(encoding='utf-8'))
assert phase07.get('phase') == '07_pinned_micdm_derived_schema_and_vocabulary_preflight'
assert phase07['gates']['pinned_imagewg_csv_checksum_verified'] is True
assert phase07['gates']['derived_empty_staging_schema_built_and_introspected'] is True
P07_SHA = file_sha(phase07_file)

In [3]:
source_candidates = [
    BASE / SOURCE_NAME,
    BASE / 'inputs' / SOURCE_NAME,
    PROJECT / 'inputs' / SOURCE_NAME,
    PROJECT / 'source_data' / SOURCE_NAME,
]
source_path = next((p for p in source_candidates if p.is_file()), None)
if source_path is None:
    from google.colab import files
    uploaded = files.upload()
    if SOURCE_NAME not in uploaded:
        raise FileNotFoundError(SOURCE_NAME)
    source_path = PRIVATE_WORK / SOURCE_NAME
    source_path.write_bytes(uploaded[SOURCE_NAME])
source_data = pd.read_csv(source_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
source_data.columns = [c.strip() for c in source_data.columns]

FEATURES = {
    'LHIPPOC': ('hippocampus', 'left'),
    'RHIPPOC': ('hippocampus', 'right'),
    'LENTORHIN': ('entorhinal_cortex', 'left'),
    'RENTORHIN': ('entorhinal_cortex', 'right'),
    'LMIDTEMP': ('middle_temporal', 'left'),
    'RMIDTEMP': ('middle_temporal', 'right'),
    'VENTRICLES': ('ventricles', 'unspecified'),
    'ICV': ('intracranial_volume', 'not_applicable'),
}
REQUIRED = {
    'RID', 'PTID', 'BL_EXAMDATE', 'M12_EXAMDATE',
    'BL_IMAGEUID', 'M12_IMAGEUID', 'BL_LONIUID', 'M12_LONIUID',
    'BL_VISCODE', 'M12_VISCODE', 'BL_VERSION', 'M12_VERSION',
    'BL_OVERALLQC', 'M12_OVERALLQC', 'BL_TEMPQC', 'M12_TEMPQC',
    'BL_VENTQC', 'M12_VENTQC',
} | {f'{t}_{f}' for t in ('BL', 'M12') for f in FEATURES}
missing_cols = sorted(REQUIRED - set(source_data.columns))
if missing_cols:
    raise ValueError('Required verified Phase03 source columns missing: '+repr(missing_cols))

source_data['RID'] = source_data['RID'].str.strip()
if source_data['RID'].eq('').any() or source_data['RID'].duplicated().any():
    raise ValueError('RID missing/duplicated; no silent deduplication')
if source_data['PTID'].str.strip().eq('').any():
    raise ValueError('PTID missing; source identity audit stopped')

integrity_issues = Counter()
for t in ('BL', 'M12'):
    d = pd.to_datetime(source_data[f'{t}_EXAMDATE'], errors='coerce')
    integrity_issues[f'{t}_invalid_date'] = int(d.isna().sum())
    source_data[f'_parsed_{t}_date'] = d
    missing_both = source_data[f'{t}_IMAGEUID'].str.strip().eq('') & source_data[f'{t}_LONIUID'].str.strip().eq('')
    integrity_issues[f'{t}_both_image_ids_missing'] = int(missing_both.sum())
integrity_issues['misordered_dates'] = int((source_data['_parsed_M12_date'] <= source_data['_parsed_BL_date']).sum())
for t in ('BL', 'M12'):
    for f in FEATURES:
        x = source_data[f'{t}_{f}'].str.strip()
        bad = x.ne('') & pd.to_numeric(x, errors='coerce').isna()
        integrity_issues[f'{t}_{f}_invalid_numeric'] = int(bad.sum())

critical = sum(v for k,v in integrity_issues.items() if k.endswith('invalid_date') or k.endswith('image_ids_missing') or k == 'misordered_dates' or k.endswith('invalid_numeric'))
if critical:
    raise ValueError('Source integrity failure; no staging load: '+str({k:v for k,v in integrity_issues.items() if v}))

In [4]:
PRIVATE_DB = PRIVATE_WORK / 'verified_source_evidence_only.sqlite'
if PRIVATE_DB.exists():
    PRIVATE_DB.unlink()
KEY = secrets.token_bytes(32)
def token(*parts):
    raw = '\x1f'.join(str(x) for x in parts).encode('utf-8')
    return hmac.new(KEY, raw, hashlib.sha256).hexdigest()

def decimal_text(raw):
    raw = str(raw).strip()
    if not raw:
        return None
    try:
        val = Decimal(raw)
    except InvalidOperation:
        raise ValueError('Invalid original source numeric field')
    if not val.is_finite():
        raise ValueError('Non-finite original source value')
    return format(val.normalize(), 'f')

conn = sqlite3.connect(PRIVATE_DB)
conn.execute('PRAGMA foreign_keys=ON')
conn.executescript('''
CREATE TABLE SOURCE_PERSON (
    person_id INTEGER PRIMARY KEY, rid_hmac TEXT NOT NULL UNIQUE, ptid_hmac TEXT NOT NULL
);
CREATE TABLE SOURCE_IMAGING_EVENT (
    imaging_event_id INTEGER PRIMARY KEY, person_id INTEGER NOT NULL REFERENCES SOURCE_PERSON(person_id),
    timepoint TEXT NOT NULL CHECK(timepoint IN ('BL','M12')),
    exam_date TEXT NOT NULL, visit_code TEXT,
    imageuid_hmac TEXT, loniuid_hmac TEXT, raw_version TEXT,
    raw_overallqc TEXT, raw_tempqc TEXT, raw_ventqc TEXT,
    verified_dicom_study_uid TEXT,
    verified_dicom_series_uid TEXT,
    extraction_model_name TEXT,
    UNIQUE(person_id,timepoint)
);
CREATE TABLE SOURCE_FEATURE_EVIDENCE (
    feature_event_id INTEGER PRIMARY KEY,
    imaging_event_id INTEGER NOT NULL REFERENCES SOURCE_IMAGING_EVENT(imaging_event_id),
    biomarker TEXT NOT NULL, laterality TEXT NOT NULL,
    original_source_column TEXT NOT NULL,
    source_numeric_text TEXT NOT NULL,
    canonical_decimal_text TEXT NOT NULL,
    unit_ucum TEXT,
    UNIQUE(imaging_event_id,original_source_column)
);
CREATE TABLE SOURCE_LONGITUDINAL_PAIR (
    person_id INTEGER PRIMARY KEY REFERENCES SOURCE_PERSON(person_id),
    baseline_event_id INTEGER NOT NULL UNIQUE REFERENCES SOURCE_IMAGING_EVENT(imaging_event_id),
    landmark_event_id INTEGER NOT NULL UNIQUE REFERENCES SOURCE_IMAGING_EVENT(imaging_event_id),
    interval_days INTEGER NOT NULL
);
CREATE TABLE SOURCE_RELIABILITY_EVIDENCE (
    person_id INTEGER PRIMARY KEY REFERENCES SOURCE_PERSON(person_id),
    source_trace_state TEXT NOT NULL,
    rule_version TEXT NOT NULL,
    calibration_reference TEXT NOT NULL
);
''')

source_count = 0
feature_count = 0
for i, row in source_data.iterrows():
    pid = i + 1
    rid = row['RID'].strip()
    conn.execute('INSERT INTO SOURCE_PERSON VALUES (?,?,?)', (pid, token('RID', rid), token('PTID', row['PTID'].strip())))
    event_ids = {}
    for t in ('BL','M12'):
        event_id = 2*i + (1 if t == 'BL' else 2)
        event_ids[t] = event_id
        img = row[f'{t}_IMAGEUID'].strip()
        loni = row[f'{t}_LONIUID'].strip()
        conn.execute('''INSERT INTO SOURCE_IMAGING_EVENT VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)''', (
            event_id, pid, t, row[f'_parsed_{t}_date'].date().isoformat(),
            row[f'{t}_VISCODE'].strip() or None,
            token('IMAGEUID', img) if img else None,
            token('LONIUID', loni) if loni else None,
            row[f'{t}_VERSION'].strip() or None,
            row[f'{t}_OVERALLQC'].strip() or None,
            row[f'{t}_TEMPQC'].strip() or None,
            row[f'{t}_VENTQC'].strip() or None,
            None, None, None,
        ))
        for f, (bio, side) in FEATURES.items():
            col = f'{t}_{f}'
            original = str(row[col]).strip()
            if not original:
                continue
            feature_count += 1
            conn.execute('''INSERT INTO SOURCE_FEATURE_EVIDENCE VALUES (?,?,?,?,?,?,?,?)''', (
                feature_count, event_id, bio, side, col,
                original, decimal_text(original), None,
            ))
    conn.execute('INSERT INTO SOURCE_LONGITUDINAL_PAIR VALUES (?,?,?,?)', (
        pid, event_ids['BL'], event_ids['M12'],
        int((row['_parsed_M12_date'] - row['_parsed_BL_date']).days),
    ))
    source_count += 1
conn.commit()

In [5]:
counts = {
    'source_persons': conn.execute('SELECT COUNT(*) FROM SOURCE_PERSON').fetchone()[0],
    'imaging_events': conn.execute('SELECT COUNT(*) FROM SOURCE_IMAGING_EVENT').fetchone()[0],
    'feature_events': conn.execute('SELECT COUNT(*) FROM SOURCE_FEATURE_EVIDENCE').fetchone()[0],
    'longitudinal_pairs': conn.execute('SELECT COUNT(*) FROM SOURCE_LONGITUDINAL_PAIR').fetchone()[0],
    'source_trace_states': conn.execute('SELECT COUNT(*) FROM SOURCE_RELIABILITY_EVIDENCE').fetchone()[0],
}
orphans = conn.execute('''
SELECT COUNT(*) FROM SOURCE_FEATURE_EVIDENCE f
LEFT JOIN SOURCE_IMAGING_EVENT i ON f.imaging_event_id = i.imaging_event_id
WHERE i.imaging_event_id IS NULL
''').fetchone()[0]
foreign_key_failures = len(conn.execute('PRAGMA foreign_key_check').fetchall())

source_decimal = {}
for i, row in source_data.iterrows():
    for t in ('BL','M12'):
        for f in FEATURES:
            col = f'{t}_{f}'
            val = str(row[col]).strip()
            if val:
                source_decimal[(2*i+(1 if t == 'BL' else 2),col)] = decimal_text(val)

reconstructed_decimal = {
    (event,col): dec
    for event,col,dec in conn.execute('''
    SELECT imaging_event_id,original_source_column,canonical_decimal_text
    FROM SOURCE_FEATURE_EVIDENCE
    ''')
}
decimal_mismatches = sum(1 for key,val in source_decimal.items() if reconstructed_decimal.get(key) != val)
missing_or_extra_events = len(set(source_decimal).symmetric_difference(reconstructed_decimal))

sql_hippo = '''
SELECT p.person_id FROM SOURCE_LONGITUDINAL_PAIR p
JOIN SOURCE_IMAGING_EVENT b ON b.imaging_event_id=p.baseline_event_id
JOIN SOURCE_IMAGING_EVENT m ON m.imaging_event_id=p.landmark_event_id
WHERE m.exam_date > b.exam_date
AND b.imageuid_hmac IS NOT NULL AND m.imageuid_hmac IS NOT NULL
AND (SELECT COUNT(DISTINCT laterality) FROM SOURCE_FEATURE_EVIDENCE f
     WHERE f.imaging_event_id=b.imaging_event_id AND biomarker='hippocampus'
       AND CAST(f.canonical_decimal_text AS REAL)>0)=2
AND (SELECT COUNT(DISTINCT laterality) FROM SOURCE_FEATURE_EVIDENCE f
     WHERE f.imaging_event_id=m.imaging_event_id AND biomarker='hippocampus'
       AND CAST(f.canonical_decimal_text AS REAL)>0)=2
'''
sql_structural_ids = {x[0] for x in conn.execute(sql_hippo)}
sql_structural_eligible = len(sql_structural_ids)
source_eligibility = (
    source_data['_parsed_M12_date'].gt(source_data['_parsed_BL_date']) &
    source_data['BL_IMAGEUID'].str.strip().ne('') &
    source_data['M12_IMAGEUID'].str.strip().ne('') &
    pd.to_numeric(source_data['BL_LHIPPOC'],errors='coerce').gt(0) &
    pd.to_numeric(source_data['BL_RHIPPOC'],errors='coerce').gt(0) &
    pd.to_numeric(source_data['M12_LHIPPOC'],errors='coerce').gt(0) &
    pd.to_numeric(source_data['M12_RHIPPOC'],errors='coerce').gt(0)
)
source_structural_ids = set((source_data.index[source_eligibility]+1).tolist())
source_structural_eligible = len(source_structural_ids)

parity = {
    **counts,
    'source_nonblank_valid_numeric_features': len(source_decimal),
    'exact_decimal_value_mismatches': decimal_mismatches,
    'missing_or_extra_feature_keys': missing_or_extra_events,
    'orphan_features': orphans,
    'foreign_key_failures': foreign_key_failures,
    'source_structural_bilateral_hippocampus_eligible': source_structural_eligible,
    'sqlite_staging_structural_bilateral_hippocampus_eligible': sql_structural_eligible,
    'cohort_membership_equivalence_for_structural_query': source_structural_ids == sql_structural_ids,
    'cohort_member_symmetric_difference': len(source_structural_ids.symmetric_difference(sql_structural_ids)),
    'approved_MCI_cohort': False,
    'verified_measurement_units': False,
    'verified_DICOM_UIDs': False,
    'actual_MI_CDM_load': False,
}
parity['private_staging_parity_pass'] = all([
    source_count == len(source_data),
    counts['imaging_events'] == 2*len(source_data),
    counts['feature_events'] == len(source_decimal),
    counts['longitudinal_pairs'] == len(source_data),
    decimal_mismatches == 0,
    missing_or_extra_events == 0,
    orphans == 0,
    foreign_key_failures == 0,
    source_structural_ids == sql_structural_ids,
])
if not parity['private_staging_parity_pass']:
    raise AssertionError('Actual-source staging parity failed: '+json.dumps(parity))
print(json.dumps(parity, indent=2))

{
  "source_persons": 258,
  "imaging_events": 516,
  "feature_events": 4128,
  "longitudinal_pairs": 258,
  "source_trace_states": 0,
  "source_nonblank_valid_numeric_features": 4128,
  "exact_decimal_value_mismatches": 0,
  "missing_or_extra_feature_keys": 0,
  "orphan_features": 0,
  "foreign_key_failures": 0,
  "source_structural_bilateral_hippocampus_eligible": 258,
  "sqlite_staging_structural_bilateral_hippocampus_eligible": 258,
  "cohort_membership_equivalence_for_structural_query": true,
  "cohort_member_symmetric_difference": 0,
  "approved_MCI_cohort": false,
  "verified_measurement_units": false,
  "verified_DICOM_UIDs": false,
  "actual_MI_CDM_load": false,
  "private_staging_parity_pass": true
}


In [6]:
SOURCE_META = PROJECT / 'source_metadata'
source_documents = []
if SOURCE_META.is_dir():
    allowed = {'.csv','.tsv','.xlsx','.pdf','.docx','.json','.txt'}
    for p in sorted(SOURCE_META.iterdir()):
        if p.is_file() and p.suffix.lower() in allowed and p.stat().st_size < 40_000_000:
            source_documents.append(p)

metadata_evidence = {
    'source_metadata_folder_present': SOURCE_META.is_dir(),
    'candidate_source_documents': len(source_documents),
    'document_extensions': dict(Counter(x.suffix.lower() for x in source_documents)),
    'exact_local_export_unit_approved': False,
    'verified_DICOM_study_uid_approved': False,
    'verified_DICOM_series_uid_approved': False,
    'original_TRACE_reliability_state_approved': False,
    'original_processing_software_identity_approved': False,
}

metadata_evidence['document_signals_for_manual_review'] = []
for index, doc in enumerate(source_documents, 1):
    text = ''
    status = 'READABLE'
    try:
        ext = doc.suffix.lower()
        if ext in {'.csv','.tsv','.txt','.json'}:
            text = doc.read_bytes()[:2_000_000].decode('utf-8-sig',errors='replace')
        elif ext in {'.docx','.xlsx'}:
            # Office XML is text-bearing; examine only metadata terminology, not participant records.
            with zipfile.ZipFile(doc) as zdoc:
                members = [n for n in zdoc.namelist() if n.endswith('.xml') and ('document.xml' in n or 'sharedStrings.xml' in n)]
                text = ' '.join(zdoc.read(n).decode('utf-8','replace')[:1_000_000] for n in members[:5])
        elif ext == '.pdf':
            try:
                from pypdf import PdfReader
                reader = PdfReader(str(doc))
                text = ' '.join((page.extract_text() or '') for page in reader.pages[:15])
            except ImportError:
                status = 'PDF_TEXT_READER_NOT_INSTALLED'
            except Exception:
                status = 'PDF_TEXT_UNAVAILABLE'
    except Exception:
        status = 'UNREADABLE'
    low = re.sub(r'<[^>]+>', ' ', text).lower()
    metadata_evidence['document_signals_for_manual_review'].append({
        'document_ref': 'source_document_'+str(index).zfill(3),
        'extension': doc.suffix.lower(),
        'read_status': status,
        'mentions_adni_longitudinal_freesurfer': any(t in low for t in ['ucsffsl','longitudinal freesurfer']),
        'mentions_export_column_names': [x for x in ['imageuid','loniuid','lhippoc','rhippoc','icv','version'] if x in low],
        'visible_unit_token_mm3': bool(re.search(r'(?<![a-z0-9])(?:mm\s*\^?\s*3|mm³|cubic millimet(?:er|re)s?)(?![a-z0-9])',low)),
    })

# Read public ADNI documentation as plain text; remove scripts, images and base64 before searching.
import urllib.request
from html.parser import HTMLParser
class VisibleText(HTMLParser):
    def __init__(self):
        super().__init__()
        self.skip = 0
        self.items = []
    def handle_starttag(self, tag, attrs):
        if tag in {'script','style','svg','noscript'}:
            self.skip += 1
    def handle_endtag(self, tag):
        if tag in {'script','style','svg','noscript'} and self.skip:
            self.skip -= 1
    def handle_data(self, data):
        if not self.skip and data.strip() and not data.strip().startswith('data:image'):
            self.items.append(data.strip())

ADNI_DOC = 'https://adni.loni.usc.edu/quick-start-guide-asset/MRI_tables.html'
public_doc = {'url': ADNI_DOC, 'downloaded': False, 'sha256': None,
              'mentions_longitudinal_freesurfer': False, 'mentions_table_specific_unit': False,
              'does_not_verify_local_export_units': True}
try:
    req = urllib.request.Request(ADNI_DOC, headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=30) as response:
        raw_doc = response.read()
    parser = VisibleText()
    parser.feed(raw_doc.decode('utf-8',errors='replace'))
    visible = ' '.join(parser.items)
    public_doc['downloaded'] = True
    public_doc['sha256'] = hashlib.sha256(raw_doc).hexdigest()
    public_doc['mentions_longitudinal_freesurfer'] = ('longitudinal freesurfer' in visible.lower())
    public_doc['mentions_table_specific_unit'] = bool(re.search(r'\b(?:mm\s*\^?\s*3|mm³|cubic millimet(?:er|re)s?)\b',visible,re.I))
except Exception as e:
    public_doc['error_type'] = type(e).__name__

In [7]:
VOCAB_DIR = PROJECT / 'vocabulary'
possible_vocab = []
for dirpath in [VOCAB_DIR, BASE]:
    if dirpath.is_dir():
        for p in dirpath.iterdir():
            if p.is_file() and (p.name.upper() == 'CONCEPT.CSV' or p.suffix.lower() == '.zip'):
                possible_vocab.append(p)
possible_vocab = sorted(set(possible_vocab), key=lambda p: (p.suffix.lower() != '.zip',str(p)))

def zip_member(z, name):
    matches = [n for n in z.namelist() if Path(n).name.upper() == name.upper()]
    return matches[0] if len(matches) == 1 else None

def read_tabular(fileobj, nrows=None, chunksize=None):
    pos = fileobj.tell() if fileobj.seekable() else None
    first = fileobj.readline().decode('utf-8-sig','replace') if isinstance(fileobj.readline(0), bytes) else fileobj.readline()
    if pos is not None:
        fileobj.seek(pos)
    sep = '\t' if first.count('\t') > first.count(',') else ','
    return pd.read_csv(fileobj, sep=sep, dtype=str, keep_default_na=False,
                       encoding='utf-8-sig', nrows=nrows, chunksize=chunksize)

vocab = {
    'snapshot_present': False,
    'candidate_concepts_only_not_approved': True,
    'package_sha256': None,
    'vocabulary_versions': [],
    'concept_rows_scanned': 0,
    'concept_candidates_by_family': {},
    'status': 'NO_ATHENA_SNAPSHOT',
}
concept_path = None
vocab_path = None
z = None
for p in possible_vocab:
    if p.suffix.lower() == '.zip':
        try:
            x = zipfile.ZipFile(p)
            c = zip_member(x, 'CONCEPT.csv')
            v = zip_member(x, 'VOCABULARY.csv')
            if c:
                z, concept_path, vocab_path = x, c, v
                vocab['package_sha256'] = file_sha(p)
                break
            x.close()
        except (zipfile.BadZipFile, OSError):
            continue
    if p.name.upper() == 'CONCEPT.CSV' and concept_path is None:
        concept_path = p
        companion = p.with_name('VOCABULARY.csv')
        vocab_path = companion if companion.is_file() else None
        vocab['package_sha256'] = file_sha(p)
        break

if concept_path is not None:
    vocab['snapshot_present'] = True
    vocab['status'] = 'CONCEPT_FOUND_VERSION_UNVERIFIED'
    if vocab_path is not None:
        handle = z.open(vocab_path) if z is not None else open(vocab_path, 'rb')
        with handle as stream:
            vd = read_tabular(stream)
        vd.columns = [x.strip().lower() for x in vd.columns]
        if {'vocabulary_id','vocabulary_version'}.issubset(vd.columns):
            wanted = vd['vocabulary_id'].isin(['SNOMED','LOINC','OMOP Extension','Nebraska Lexicon','UCUM'])
            vocab['vocabulary_versions'] = vd.loc[wanted,['vocabulary_id','vocabulary_version']].to_dict('records')
            vocab['version_file_present'] = True
        else:
            vocab['version_file_present'] = False
    else:
        vocab['version_file_present'] = False

    pattern = {
        'MRI': re.compile(r'(?:magnetic resonance|\bMRI\b)',re.I),
        'hippocampus': re.compile(r'hippocamp',re.I),
        'entorhinal': re.compile(r'entorhin',re.I),
        'middle_temporal': re.compile(r'middle temporal',re.I),
        'ventricles': re.compile(r'ventric',re.I),
        'intracranial_volume': re.compile(r'intracranial volume',re.I),
    }
    candidates = {k:[] for k in pattern}
    candidate_counts = {k:0 for k in pattern}
    stream = z.open(concept_path) if z is not None else open(concept_path, 'rb')
    with stream:
        chunks = read_tabular(stream, chunksize=100_000)
        for chunk in chunks:
            chunk.columns = [x.strip().lower() for x in chunk.columns]
            required_vocab = {'concept_id','concept_name','vocabulary_id','domain_id','standard_concept','invalid_reason'}
            if not required_vocab.issubset(chunk.columns):
                vocab['status'] = 'INVALID_CONCEPT_COLUMNS'
                vocab['missing_columns'] = sorted(required_vocab - set(chunk.columns))
                break
            vocab['concept_rows_scanned'] += len(chunk)
            valid = chunk['invalid_reason'].eq('') & chunk['standard_concept'].eq('S')
            subset = chunk.loc[valid, ['concept_id','concept_name','vocabulary_id','domain_id','standard_concept']]
            for family, regex in pattern.items():
                matched = subset[subset['concept_name'].str.contains(regex, na=False, regex=True)]
                candidate_counts[family] += len(matched)
                if len(candidates[family]) < 12:
                    candidates[family].extend(matched.head(12-len(candidates[family])).to_dict('records'))
    vocab['concept_candidates_by_family'] = {
        fam: {'matching_active_standard_concepts':candidate_counts[fam],
              'examples_for_manual_mapping_review': candidates[fam]}
        for fam in pattern
    }
    if vocab['status'] != 'INVALID_CONCEPT_COLUMNS':
        vocab['status'] = 'ATHENA_CONCEPT_AND_VERSION_PRESENT' if vocab['version_file_present'] else 'CONCEPT_SCANNED_VERSION_UNVERIFIED'
if z is not None:
    z.close()

In [8]:
micdm_required = phase07['micdm_contract']
readiness = {
    'image_occurrence': {
        'person_id': 'LOCAL_PARTICIPANT_MAPPING_AVAILABLE',
        'image_occurrence_date': 'SOURCE_SCAN_DATE_AVAILABLE',
        'procedure_occurrence_id': 'NO_VERIFIED_OMOP_PROCEDURE_MAPPING',
        'image_study_UID': 'RAW_IMAGEUID_IS_NOT_VERIFIED_DICOM_STUDY_UID',
        'image_series_UID': 'NO_VERIFIED_DICOM_SERIES_UID',
        'modality_concept_id': 'NO_APPROVED_CONCEPT',
    },
    'image_feature': {
        'person_id': 'LOCAL_PARTICIPANT_MAPPING_AVAILABLE',
        'image_occurrence_id': 'PRIVATE_EVIDENCE_EVENT_LINK_AVAILABLE_NOT_OFFICIAL_IMAGE_OCCURRENCE_ID',
        'image_feature_concept_id': 'NO_APPROVED_CONCEPT',
        'image_feature_type_concept_id': 'NO_APPROVED_CONCEPT',
        'numeric_value': 'PINNED_IMAGE_FEATURE_HAS_NO_NUMERIC_VALUE_COLUMN',
        'quantity_unit': 'EXACT_LOCAL_EXPORT_UNIT_UNVERIFIED',
        'trace_state': 'SOURCE_TRACE_STATE_NOT_IN_VERIFIED_SOURCE_CSV',
    },
}
assert all(x in micdm_required for x in ['IMAGE_OCCURRENCE','IMAGE_FEATURE'])

report = {
    'project': 'ICHI2027_neuroimaging_fhir_omop_fidelity',
    'phase': '08_real_private_evidence_staging_and_vocabulary_crosswalk',
    'run_utc': now(),
    'scientific_result_status': 'REAL_SOURCE_INTERNAL_STAGING_ONLY_NO_MI_CDM_LOAD_NO_ETL_COMPARISON',
    'privacy': 'Source data and private SQLite remain in this Colab runtime; shareable JSON contains aggregates and public Athena concept candidates only.',
    'internal_pseudonymization': 'HMAC secret is session-local, never exported; internal tokens change on restart',
    'phase07_json_sha256': P07_SHA,
    'input_schema': {'source_filename_expected': SOURCE_NAME,
                     'rows': len(source_data),
                     'source_columns': len(source_data.columns)-2,
                     'source_date_and_identifier_integrity_issue_counts': dict(integrity_issues)},
    'private_real_source_evidence_staging': parity,
    'source_provenance': metadata_evidence,
    'official_adni_public_reference': public_doc,
    'athena_vocabulary': vocab,
    'pinned_micdm_staging_contract': {
        'image_occurrence_required_fields': micdm_required['IMAGE_OCCURRENCE']['required_fields'],
        'image_feature_required_fields': micdm_required['IMAGE_FEATURE']['required_fields'],
        'numeric_feature_value_column_in_spec': phase07['pinned_image_feature_numeric_value_column_present'],
    },
    'source_to_target_blockers': readiness,
    'gates': {
        'real_private_staging_parity_verified': parity['private_staging_parity_pass'],
        'real_structural_candidate_cohort_query_equal': parity['cohort_membership_equivalence_for_structural_query'],
        'baseline_MCI_status_verified': False,
        'exact_export_measurement_unit_verified': False,
        'verified_DICOM_study_and_series_UIDs': False,
        'original_TRACE_reliability_state_verified': False,
        'omop_vocab_snapshot_loaded': vocab['snapshot_present'],
        'omop_vocab_version_file_present': vocab.get('version_file_present',False),
        'omop_mappings_approved': False,
        'real_MI_CDM_load_completed': False,
        'real_TFL_MRI_adapter_executed': False,
        'real_four_arm_comparison_completed': False,
    },
    'real_four_arm_scores': None,
    'real_MI_CDM_load_results': None,
    'next_stage_gate': 'REVIEW_SOURCE_EXPORT_DICTIONARY_DICOM_IDENTIFIERS_AND_ATHENA_CONCEPTS_BEFORE_OFFICIAL_MI_CDM_LOAD',
}
SHAREABLE.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
conn.close()
print(json.dumps({'output':str(SHAREABLE),
                  'real_source_staging_pass':parity['private_staging_parity_pass'],
                  'vocabulary':vocab['status'],
                  'MI_CDM_load':False},indent=2))

{
  "output": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase08_outputs/ICHI2027_Phase08_SHAREABLE.json",
  "real_source_staging_pass": true,
  "vocabulary": "NO_ATHENA_SNAPSHOT",
  "MI_CDM_load": false
}


In [9]:
try:
    from google.colab import files
    files.download(str(SHAREABLE))
except ImportError:
    print(SHAREABLE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>